# Load data và xử lý

In [1]:
import pandas as pd

In [2]:
df_train = pd.read_json('/content/drive/MyDrive/DaiHoc/HK7/DS201/ThucHanh/Lab4/Machine-translation-datasets/PhoMT/train.json')
df_dev = pd.read_json('/content/drive/MyDrive/DaiHoc/HK7/DS201/ThucHanh/Lab4/Machine-translation-datasets/PhoMT/dev.json')
df_test = pd.read_json('/content/drive/MyDrive/DaiHoc/HK7/DS201/ThucHanh/Lab4/Machine-translation-datasets/PhoMT/test.json')

In [3]:
df_train.head()

,english,vietnamese
0,It begins with a countdown .,Câu chuyện bắt đầu với buổi lễ đếm ngược .
1,"On August 14th , 1947 , a woman in Bombay goes...","Ngày 14 , tháng 8 , năm 1947 , gần nửa đêm , ở..."
2,"Across India , people hold their breath for th...","Cùng lúc , trên khắp đất Ấn , người ta nín thở..."
3,"And at the stroke of midnight , a squirming in...","Khi đồng hồ điểm thời khắc nửa đêm , một đứa t..."
4,"These events form the foundation of "" Midnight...","Những sự kiện này là nền móng tạo nên "" Những ..."


In [4]:
print(len(df_train), len(df_dev), len(df_test))
df_train.head()

2977999 18719 19151


,english,vietnamese
0,It begins with a countdown .,Câu chuyện bắt đầu với buổi lễ đếm ngược .
1,"On August 14th , 1947 , a woman in Bombay goes...","Ngày 14 , tháng 8 , năm 1947 , gần nửa đêm , ở..."
2,"Across India , people hold their breath for th...","Cùng lúc , trên khắp đất Ấn , người ta nín thở..."
3,"And at the stroke of midnight , a squirming in...","Khi đồng hồ điểm thời khắc nửa đêm , một đứa t..."
4,"These events form the foundation of "" Midnight...","Những sự kiện này là nền móng tạo nên "" Những ..."


In [5]:
import re

def normalize_text(s):
    s = s.lower().strip()
    s = re.sub(r"\s+", " ", s)
    return s

df_train["english"] = df_train["english"].map(normalize_text)
df_train["vietnamese"] = df_train["vietnamese"].map(normalize_text)

df_dev["english"] = df_dev["english"].map(normalize_text)
df_dev["vietnamese"] = df_dev["vietnamese"].map(normalize_text)

df_test["english"] = df_test["english"].map(normalize_text)
df_test["vietnamese"] = df_test["vietnamese"].map(normalize_text)


In [6]:
from tensorflow.keras.preprocessing.text import Tokenizer

VOCAB_EN = 40000
VOCAB_VI = 40000

tokenizer_en = Tokenizer(num_words=VOCAB_EN, oov_token="<unk>")
tokenizer_vi = Tokenizer(num_words=VOCAB_VI, oov_token="<unk>")

tokenizer_en.fit_on_texts(df_train["english"])
tokenizer_vi.fit_on_texts(df_train["vietnamese"])

In [7]:
def percentile_len(texts, q=0.95):
    lens = texts.str.split().str.len()
    return int(lens.quantile(q))

max_len_en = percentile_len(df_train["english"], 0.95)
max_len_vi = percentile_len(df_train["vietnamese"], 0.95) + 2  # sos + eos

print("max_len_en:", max_len_en)
print("max_len_vi:", max_len_vi)

max_len_en: 38
max_len_vi: 49


In [8]:
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences

def seq2seq_generator(df, tokenizer_en, tokenizer_vi):
    for en, vi in zip(df["english"], df["vietnamese"]):
        # encoder input
        en_ids = tokenizer_en.texts_to_sequences([en])[0]
        en_ids = pad_sequences([en_ids], maxlen=max_len_en, padding="post")[0]

        # decoder input (<sos>)
        vi_in  = tokenizer_vi.texts_to_sequences(["<sos> " + vi])[0]
        vi_in  = pad_sequences([vi_in], maxlen=max_len_vi, padding="post")[0]

        # decoder output (<eos>)
        vi_out = tokenizer_vi.texts_to_sequences([vi + " <eos>"])[0]
        vi_out = pad_sequences([vi_out], maxlen=max_len_vi, padding="post")[0]

        yield (
            (en_ids.astype("int32"), vi_in.astype("int32")),
            vi_out.astype("int32")
        )


In [9]:
def make_dataset(df, shuffle=False, batch_size=128):
    ds = tf.data.Dataset.from_generator(
        lambda: seq2seq_generator(df, tokenizer_en, tokenizer_vi),
        output_signature=(
            (
                tf.TensorSpec(shape=(max_len_en,), dtype=tf.int32),
                tf.TensorSpec(shape=(max_len_vi,), dtype=tf.int32),
            ),
            tf.TensorSpec(shape=(max_len_vi,), dtype=tf.int32),
        )
    )
    if shuffle:
        ds = ds.shuffle(10000)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(df_train, shuffle=True)
dev_ds   = make_dataset(df_dev)
test_ds  = make_dataset(df_test)


# Bài 1.

In [10]:
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("mixed_float16")


In [11]:
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense
from tensorflow.keras.models import Model

EMB = 300
HID = 256
LAYERS = 5

# Encoder
enc_in = Input(shape=(max_len_en,))
x = Embedding(VOCAB_EN, EMB, mask_zero=True)(enc_in)

enc_states = []
for i in range(LAYERS):
    x, h, c = LSTM(HID, return_sequences=True, return_state=True)(x)
    enc_states.append((h, c))

# Decoder
dec_in = Input(shape=(max_len_vi,))
y = Embedding(VOCAB_VI, EMB, mask_zero=True)(dec_in)

for i in range(LAYERS):
    y, _, _ = LSTM(
        HID,
        return_sequences=True,
        return_state=True,
        name=f"dec_lstm_{i+1}"
    )(
        y,
        initial_state=enc_states[i]
    )

dec_out = Dense(VOCAB_VI, activation="softmax")(y)

model = Model([enc_in, dec_in], dec_out)


In [12]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy"
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 38)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 49)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 38, 300)   │ 12,000,000 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 38)        │          0 │ input_layer[0][0] │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 49, 300)   │ 12,000,000 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 38, 256), │    570,368 │ embedding[0][0],  │
│                     │ (None, 256),      │            │ not_equal[0][0]   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_lstm_1 (LSTM)   │ [(None, 49, 256), │    570,368 │ embedding_1[0][0… │
│                     │ (None, 256),      │            │ lstm[0][1],       │
│                     │ (None, 256)]      │            │ lstm[0][2]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, 38, 256), │    525,312 │ lstm[0][0],       │
│                     │ (None, 256),      │            │ not_equal[0][0]   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_lstm_2 (LSTM)   │ [(None, 49, 256), │    525,312 │ dec_lstm_1[0][0], │
│                     │ (None, 256),      │            │ lstm_1[0][1],     │
│                     │ (None, 256)]      │            │ lstm_1[0][2]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ [(None, 38, 256), │    525,312 │ lstm_1[0][0],     │
│                     │ (None, 256),      │            │ not_equal[0][0]   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_lstm_3 (LSTM)   │ [(None, 49, 256), │    525,312 │ dec_lstm_2[0][0], │
│                     │ (None, 256),      │            │ lstm_2[0][1],     │
│                     │ (None, 256)]      │            │ lstm_2[0][2]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_3 (LSTM)       │ [(None, 38, 256), │    525,312 │ lstm_2[0][0],     │
│                     │ (None, 256),      │            │ not_equal[0][0]   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_lstm_4 (LSTM)   │ [(None, 49, 256), │    525,312 │ dec_lstm_3[0][0], │
│                     │ (None, 256),      │            │ lstm_3[0][1],     │
│                     │ (None, 256)]      │            │ lstm_3[0][2]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_4 (LSTM)       │ [(None, 38, 256), │    525,312 │ lstm_3[0][0],     │
│                     │ (None, 256),      │            │ not_equal[0][0] 

 Total params: 39,623,232 (151.15 MB)

 Trainable params: 39,623,232 (151.15 MB)

 Non-trainable params: 0 (0.00 B)

In [13]:
num_train = len(df_train)
num_dev   = len(df_dev)

BATCH_SIZE = 128

steps_per_epoch = num_train // BATCH_SIZE
val_steps = num_dev // BATCH_SIZE

In [14]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

ckpt = ModelCheckpoint(
    "seq2seq_best.weights.h5",
    save_best_only=True,
    save_weights_only=True
)

early = EarlyStopping(patience=3, restore_best_weights=True)


In [15]:
model.fit(
    train_ds,
    validation_data=dev_ds,
    epochs=2,
    steps_per_epoch=steps_per_epoch,
    validation_steps=val_steps,
    callbacks=[ckpt, early]
)

Epoch 1/2
23265/23265 ━━━━━━━━━━━━━━━━━━━━ 7719s 331ms/step - loss: 4.7560 - val_loss: 3.5295
Epoch 2/2
    1/23265 ━━━━━━━━━━━━━━━━━━━━ 1:39:34 257ms/step - loss: 3.1083

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


23265/23265 ━━━━━━━━━━━━━━━━━━━━ 23s 984us/step - loss: 3.1083 - val_loss: 3.5240


In [16]:
import tensorflow as tf
import numpy as np
import re
from tqdm import tqdm
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score

smooth = SmoothingFunction().method1


In [17]:
def greedy_decode_batch(model, enc_in, max_len):
    B = enc_in.shape[0]

    # decoder input luôn có shape (B, max_len)
    dec_in = tf.zeros((B, max_len), dtype=tf.int32)

    outputs = []

    for t in range(max_len):
        logits = model([enc_in, dec_in], training=False)

        # lấy token tại vị trí t
        next_id = tf.argmax(logits[:, t, :], axis=-1, output_type=tf.int32)

        outputs.append(next_id)

        # ghi token vào dec_in tại vị trí t
        dec_in = tf.tensor_scatter_nd_update(
            dec_in,
            indices=tf.stack(
                [tf.range(B), tf.fill([B], t)], axis=1
            ),
            updates=next_id
        )

    return tf.stack(outputs, axis=1)  # (B, max_len)


In [18]:
y_true_texts = []
y_pred_texts = []

for (enc_in, _), dec_out in test_ds:
    preds = greedy_decode_batch(model, enc_in, max_len_vi)

    for gt, pr in zip(dec_out.numpy(), preds.numpy()):
        gt = [x for x in gt if x != 0]
        pr = [x for x in pr if x != 0]

        y_true_texts.append(tokenizer_vi.sequences_to_texts([gt])[0])
        y_pred_texts.append(tokenizer_vi.sequences_to_texts([pr])[0])


In [19]:
def norm(s):
    s = s.lower().strip()
    s = re.sub(r"\s+", " ", s)
    return s

refs = [norm(x) for x in y_true_texts]
hyps = [norm(x) for x in y_pred_texts]

In [20]:
def bleu_scores(refs, hyps):
    b1=b2=b3=b4=0
    for r,h in zip(refs,hyps):
        r,h = r.split(), h.split()
        b1+=sentence_bleu([r],h,(1,0,0,0),smooth)
        b2+=sentence_bleu([r],h,(.5,.5,0,0),smooth)
        b3+=sentence_bleu([r],h,(1/3,)*3+(0,),smooth)
        b4+=sentence_bleu([r],h,(.25,)*4,smooth)
    n=len(refs)
    return b1/n,b2/n,b3/n,b4/n


In [21]:
def lcs(a,b):
    dp=[[0]*(len(b)+1) for _ in range(len(a)+1)]
    for i in range(len(a)):
        for j in range(len(b)):
            dp[i+1][j+1]=dp[i][j]+1 if a[i]==b[j] else max(dp[i][j+1],dp[i+1][j])
    return dp[-1][-1]

def rouge_l(refs, hyps):
    s=0
    for r,h in zip(refs,hyps):
        r,h=r.split(),h.split()
        l=lcs(r,h)
        p=l/max(len(h),1); r_=l/max(len(r),1)
        s+=0 if p+r_==0 else 2*p*r_/(p+r_)
    return s/len(refs)


In [23]:
def compute_meteor(refs, hyps):
    scores = []
    for r, h in zip(refs, hyps):
        scores.append(meteor_score([r], h))
    return sum(scores) / len(scores)

In [22]:
b1,b2,b3,b4 = bleu_scores(refs,hyps)
print(f"BLEU@1 : {b1:.4f}")
print(f"BLEU@2 : {b2:.4f}")
print(f"BLEU@3 : {b3:.4f}")
print(f"BLEU@4 : {b4:.4f}")
print(f"ROUGE-L: {rouge_l(refs,hyps):.4f}")


BLEU@1 : 0.0885
BLEU@2 : 0.0254
BLEU@3 : 0.0111
BLEU@4 : 0.0071
ROUGE-L: 0.1125
